# 📊 NIFTY50 Quantile Forecasting — LSTM with Full Hyperparameter Tuning

## Research Paper References

| # | Title | Authors | Journal |
|---|-------|---------|---------|
| 1 | Deep Quantile Regression | Rodrigues & Pereira (2020) | *International Journal of Forecasting* |
| 2 | Probabilistic Forecasting with LSTMs (DeepAR) | Salinas et al. (2020) | *NeurIPS* |
| 3 | Conformal Prediction for Time Series | Stankeviciute et al. (2021) | *NeurIPS* |
| 4 | Uncertainty Quantification via Quantile Regression | Koenker & Bassett (1978) | *Econometrica* |

### Key Design Choices:
- **Grid Search** over Window Size × LSTM Units × Dropout × Learning Rate
- **Pinball (Quantile) Loss** for each quantile head
- **Quantiles**: Q05, Q10, Q25, Q50 (median), Q75, Q90, Q95
- **Metrics**: PICP (coverage), PINAW (sharpness), Winkler Score, Median RMSE
- Best model auto-selected by median-RMSE on validation set
- **Plots**: Interval fan chart, Coverage calibration, 5-day probabilistic forecast

---


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (LSTM, Dense, Dropout, BatchNormalization,
                                     Input)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K
import math, itertools, time

np.random.seed(42)
tf.random.set_seed(42)

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size']      = 12
plt.rcParams['axes.grid']      = True
plt.rcParams['grid.alpha']     = 0.3
plt.style.use('seaborn-v0_8-darkgrid')

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print("\n✅ Environment ready.")


## 1. Configuration & Hyperparameter Grid

In [ ]:
DATA_PATH      = "../nifty_final_dataset.csv"
TARGET_COL     = "target"
DATE_COL       = "date"
FORECAST_STEPS = 5

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

BATCH_SIZE = 32
EPOCHS     = 100
PATIENCE   = 15

QUANTILES = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
Q_LABELS  = [f"Q{int(q*100):02d}" for q in QUANTILES]

PARAM_GRID = {
    "window_size"   : [5, 10, 20],
    "units"         : [32, 64, 128],
    "dropout"       : [0.0, 0.2, 0.3],
    "learning_rate" : [0.001, 0.0005],
}

all_combos = list(itertools.product(
    PARAM_GRID["window_size"],
    PARAM_GRID["units"],
    PARAM_GRID["dropout"],
    PARAM_GRID["learning_rate"],
))

print(f"Quantiles              : {QUANTILES}")
print(f"Total combinations     : {len(all_combos)}")
for k, v in PARAM_GRID.items():
    print(f"  {k:20s}: {v}")


## 2. Load & Inspect Data

In [ ]:
df = pd.read_csv(DATA_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)
df = df.dropna(subset=[TARGET_COL])

print(f"Dataset shape  : {df.shape}")
print(f"Date range     : {df[DATE_COL].min().date()}  →  {df[DATE_COL].max().date()}")
print(f"\nTarget statistics:")
print(df[TARGET_COL].describe().round(6))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(df[DATE_COL], df[TARGET_COL], lw=0.8, color='steelblue')
axes[0].set_title('NIFTY50 Daily Returns'); axes[0].set_ylabel('Return')

axes[1].hist(df[TARGET_COL], bins=80, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', ls='--', lw=1.5, label='Zero')
axes[1].set_title('Return Distribution')
axes[1].legend()

plt.suptitle('NIFTY50 Data Overview', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 3. Helpers — Pinball Loss, Model, Data Pipeline

In [ ]:
def pinball_loss(quantile):
    """Keras-compatible pinball (quantile) loss for a given quantile."""
    def loss(y_true, y_pred):
        e = y_true - y_pred
        return K.mean(K.maximum(quantile * e, (quantile - 1) * e))
    loss.__name__ = f"pinball_q{int(quantile*100):02d}"
    return loss


def build_quantile_model(n_features, n_quantiles, units, dropout, lr):
    """Shared LSTM backbone → one Dense output head per quantile."""
    inp = Input(shape=(None, n_features))
    x   = LSTM(units, return_sequences=True)(inp)
    x   = BatchNormalization()(x)
    x   = Dropout(dropout)(x)
    x   = LSTM(units // 2, return_sequences=False)(x)
    x   = BatchNormalization()(x)
    x   = Dropout(dropout)(x)
    x   = Dense(units // 4, activation='relu')(x)
    outputs = [Dense(1, name=f"q{int(q*100):02d}")(x) for q in QUANTILES]
    model   = Model(inputs=inp, outputs=outputs)

    losses  = {f"q{int(q*100):02d}": pinball_loss(q) for q in QUANTILES}
    model.compile(optimizer=Adam(lr), loss=losses)
    return model


def build_sequences(X_arr, y_arr, window):
    Xs, ys = [], []
    for i in range(len(X_arr) - window):
        Xs.append(X_arr[i:i+window])
        ys.append(y_arr[i+window])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)


def split_and_scale(df, window, target_col, date_col, train_r=0.70, val_r=0.15):
    feat_cols = [c for c in df.columns if c not in [target_col, date_col]]
    X_raw = df[feat_cols].values
    y_raw = df[target_col].values
    dates = df[date_col].values
    n = len(df)
    nt, nv = int(n*train_r), int(n*val_r)

    X_tr, y_tr = X_raw[:nt],       y_raw[:nt]
    X_va, y_va = X_raw[nt:nt+nv],  y_raw[nt:nt+nv]
    X_te, y_te = X_raw[nt+nv:],    y_raw[nt+nv:]
    d_te        = dates[nt+nv:]

    sc = RobustScaler()
    X_tr = sc.fit_transform(X_tr)
    X_va = sc.transform(X_va)
    X_te = sc.transform(X_te)

    Xs_tr, ys_tr = build_sequences(X_tr, y_tr, window)
    Xs_va, ys_va = build_sequences(X_va, y_va, window)
    Xs_te, ys_te = build_sequences(X_te, y_te, window)
    return (Xs_tr, ys_tr, Xs_va, ys_va, Xs_te, ys_te, d_te[window:], sc, X_te)


def quantile_metrics(y_true, preds_dict, quantiles, low_q=0.05, high_q=0.95):
    """PICP, PINAW, Winkler, Median RMSE, Median MAE."""
    med = preds_dict[0.50].flatten()
    med_rmse = math.sqrt(mean_squared_error(y_true, med))
    med_mae  = mean_absolute_error(y_true, med)

    lo  = preds_dict[low_q].flatten()
    hi  = preds_dict[high_q].flatten()
    alpha = high_q - low_q
    picp  = np.mean((y_true >= lo) & (y_true <= hi)) * 100
    pinaw = np.mean(hi - lo) / (y_true.max() - y_true.min() + 1e-9)

    # Winkler score
    winkler = np.mean(
        (hi - lo) +
        (2/alpha) * np.maximum(lo - y_true, 0) +
        (2/alpha) * np.maximum(y_true - hi, 0)
    )

    # Coverage per quantile
    coverages = {}
    for q in quantiles:
        pred_q = preds_dict[q].flatten()
        coverages[q] = np.mean(y_true <= pred_q)

    return dict(
        med_RMSE=med_rmse, med_MAE=med_mae,
        PICP=picp, PINAW=pinaw, Winkler=winkler,
        coverages=coverages
    )

print("✅ Helpers defined.")


## 4. Hyperparameter Grid Search

In [ ]:
feature_cols = [c for c in df.columns if c not in [TARGET_COL, DATE_COL]]
n_features   = len(feature_cols)
n_quantiles  = len(QUANTILES)

results, total = [], len(all_combos)
print(f"Starting quantile grid search over {total} combinations ...\n")

for idx, (ws, units, dropout, lr) in enumerate(all_combos, 1):
    t0 = time.time()

    (Xs_tr, ys_tr, Xs_va, ys_va,
     Xs_te, ys_te, ds_te, sc, X_te_sc) = split_and_scale(df, ws, TARGET_COL, DATE_COL)

    model = build_quantile_model(n_features, n_quantiles, units, dropout, lr)

    cbs = [
        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, verbose=0)
    ]

    # Build target dict for multi-output model
    ys_tr_dict = {f"q{int(q*100):02d}": ys_tr for q in QUANTILES}
    ys_va_dict = {f"q{int(q*100):02d}": ys_va for q in QUANTILES}

    history = model.fit(
        Xs_tr, ys_tr_dict,
        validation_data=(Xs_va, ys_va_dict),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=cbs, verbose=0
    )

    # Predict on validation
    val_raw = model.predict(Xs_va, verbose=0)
    val_preds = {q: (val_raw[i] if n_quantiles > 1 else val_raw)
                 for i, q in enumerate(QUANTILES)}
    vm = quantile_metrics(ys_va, val_preds, QUANTILES)

    elapsed = time.time() - t0
    results.append(dict(
        window_size=ws, units=units, dropout=dropout, learning_rate=lr,
        val_med_RMSE=vm['med_RMSE'], val_med_MAE=vm['med_MAE'],
        val_PICP=vm['PICP'], val_PINAW=vm['PINAW'], val_Winkler=vm['Winkler'],
        epochs_run=len(history.history['loss']), time_sec=round(elapsed,1)
    ))

    print(f"[{idx:>3}/{total}]  ws={ws:>2}  units={units:>3}  dropout={dropout:.1f}  lr={lr:.4f}  "          f"→ medRMSE={vm['med_RMSE']:.6f}  PICP={vm['PICP']:.1f}%  Winkler={vm['Winkler']:.5f}  ({elapsed:.1f}s)")

results_df = pd.DataFrame(results).sort_values('val_med_RMSE').reset_index(drop=True)
print("\n✅ Grid search complete.")


## 5. All Results — Sorted by Validation Median RMSE

In [ ]:
print("=" * 100)
print(f"{'Rank':>4}  {'ws':>4}  {'units':>5}  {'drop':>5}  {'lr':>7}  "      f"{'medRMSE':>9}  {'medMAE':>9}  {'PICP%':>7}  {'PINAW':>7}  {'Winkler':>9}  {'epochs':>6}")
print("=" * 100)
for i, r in results_df.iterrows():
    m = " ◄ BEST" if i == 0 else ""
    print(f"{i+1:>4}  {int(r.window_size):>4}  {int(r.units):>5}  {r.dropout:>5.2f}  "          f"{r.learning_rate:>7.4f}  {r.val_med_RMSE:>9.6f}  {r.val_med_MAE:>9.6f}  "          f"{r.val_PICP:>7.2f}  {r.val_PINAW:>7.4f}  {r.val_Winkler:>9.5f}  {int(r.epochs_run):>6}{m}")
print("=" * 100)

# Heatmap
pivot = results_df.groupby(['window_size','units'])['val_med_RMSE'].mean().unstack()
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.6f', cmap='YlOrRd_r', linewidths=0.5,
            ax=ax, cbar_kws={'label':'Mean val Median RMSE'})
ax.set_title('Mean Val Median RMSE — Window vs Units', fontweight='bold')
plt.tight_layout(); plt.show()

# PICP heatmap
pivot_picp = results_df.groupby(['window_size','units'])['val_PICP'].mean().unstack()
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot_picp, annot=True, fmt='.1f', cmap='Blues', linewidths=0.5,
            ax=ax, cbar_kws={'label':'Mean PICP (90% interval, %)'},
            vmin=80, vmax=100)
ax.set_title('Coverage (PICP %) — Window vs Units', fontweight='bold')
plt.tight_layout(); plt.show()


## 6. Best Hyperparameters (Auto-Selected)

In [ ]:
best = results_df.iloc[0]
BEST_WINDOW  = int(best['window_size'])
BEST_UNITS   = int(best['units'])
BEST_DROPOUT = float(best['dropout'])
BEST_LR      = float(best['learning_rate'])

print("╔" + "═"*54 + "╗")
print("║" + " BEST HYPERPARAMETERS ".center(54) + "║")
print("╠" + "═"*54 + "╣")
print(f"║  Window Size      : {BEST_WINDOW:<35}║")
print(f"║  LSTM Units       : {BEST_UNITS:<35}║")
print(f"║  Dropout          : {BEST_DROPOUT:<35}║")
print(f"║  Learning Rate    : {BEST_LR:<35}║")
print(f"║  Val Median RMSE  : {best['val_med_RMSE']:<35.6f}║")
print(f"║  Val PICP (90%)   : {best['val_PICP']:<33.2f} % ║")
print(f"║  Val Winkler      : {best['val_Winkler']:<35.5f}║")
print("╚" + "═"*54 + "╝")


## 7. Retrain Final Quantile Model

In [ ]:
(Xs_tr, ys_tr, Xs_va, ys_va,
 Xs_te, ys_te, ds_te, sc, X_te_sc) = split_and_scale(
     df, BEST_WINDOW, TARGET_COL, DATE_COL)

final_model = build_quantile_model(n_features, len(QUANTILES), BEST_UNITS, BEST_DROPOUT, BEST_LR)
final_model.summary()

ys_tr_dict = {f"q{int(q*100):02d}": ys_tr for q in QUANTILES}
ys_va_dict = {f"q{int(q*100):02d}": ys_va for q in QUANTILES}

cbs_final = [
    EarlyStopping(monitor='val_loss', patience=20,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=8, min_lr=1e-6, verbose=1)
]

history_final = final_model.fit(
    Xs_tr, ys_tr_dict,
    validation_data=(Xs_va, ys_va_dict),
    epochs=150, batch_size=BATCH_SIZE,
    callbacks=cbs_final, verbose=1
)
print("\n✅ Final model training complete.")


## 8. Test Set Evaluation

In [ ]:
def predict_quantiles(model, X):
    raw = model.predict(X, verbose=0)
    return {q: raw[i] for i, q in enumerate(QUANTILES)}

test_preds = predict_quantiles(final_model, Xs_te)
val_preds2 = predict_quantiles(final_model, Xs_va)
tr_preds2  = predict_quantiles(final_model, Xs_tr)

test_m = quantile_metrics(ys_te, test_preds, QUANTILES)
val_m2 = quantile_metrics(ys_va, val_preds2, QUANTILES)
tr_m2  = quantile_metrics(ys_tr, tr_preds2,  QUANTILES)

print("\n" + "═"*65)
print(" FINAL QUANTILE MODEL — METRICS ".center(65))
print("═"*65)
print(f"{'Metric':>15}  {'Train':>12}  {'Validation':>12}  {'Test':>12}")
print("-"*65)
for key in ['med_RMSE','med_MAE','PICP','PINAW','Winkler']:
    print(f"{key:>15}  {tr_m2[key]:>12.6f}  {val_m2[key]:>12.6f}  {test_m[key]:>12.6f}")
print("═"*65)

print("\nQuantile Coverage Calibration (Test Set):")
print(f"  {'Quantile':>10}  {'Expected %':>12}  {'Observed %':>12}  {'Gap':>8}")
for q in QUANTILES:
    exp = q * 100
    obs = test_m['coverages'][q] * 100
    gap = obs - exp
    print(f"  {q:>10.2f}  {exp:>12.1f}  {obs:>12.1f}  {gap:>+8.1f}")

# ── Directional accuracy on median ──
med_pred = test_preds[0.50].flatten()
dir_acc  = np.mean(np.sign(ys_te) == np.sign(med_pred)) * 100
print(f"\nMedian Directional Accuracy (test): {dir_acc:.2f}%")

# ── Interpretation ──
print("\n" + "═"*65)
print(" INTERPRETATION ".center(65))
print("═"*65)

picp = test_m['PICP']
if picp >= 87:
    cov_note = f"✅ GOOD coverage: PICP={picp:.1f}% (target 90%)"
elif picp >= 80:
    cov_note = f"⚠️  MARGINAL coverage: PICP={picp:.1f}% (target 90%)"
else:
    cov_note = f"❌ POOR coverage: PICP={picp:.1f}% (target 90%)"

rmse = test_m['med_RMSE']
if rmse < 0.005:
    rmse_note = "✅ EXCELLENT median RMSE for financial returns"
elif rmse < 0.010:
    rmse_note = "✅ GOOD median RMSE"
elif rmse < 0.015:
    rmse_note = "⚠️  MODERATE — use with caution"
else:
    rmse_note = "❌ POOR — reconsider features/architecture"

if dir_acc > 55:
    rel_note = "🟢 Median forecast RELIABLE for directional signals"
elif dir_acc > 50:
    rel_note = "🟡 Marginal directional signal — trade with caution"
else:
    rel_note = "🔴 Directional signal UNRELIABLE"

print(f"  {cov_note}")
print(f"  {rmse_note}")
print(f"  {rel_note}")
print(f"  PINAW={test_m['PINAW']:.4f} — {'Narrow intervals' if test_m['PINAW'] < 0.5 else 'Wide intervals (high uncertainty)'}")
print("═"*65)


## 9. Diagnostic Plots

In [ ]:
# ── Plot 1: Training Loss ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(history_final.history['loss'],     label='Train Total Loss', lw=2)
ax.plot(history_final.history['val_loss'], label='Val Total Loss',   lw=2, ls='--')
ax.set_title('Quantile Model — Training & Validation Loss', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Total Pinball Loss')
ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── Plot 2: Quantile Fan Chart — Test Set ────────────────────────────
dates_dt = pd.to_datetime(ds_te)

colors_lo = ['#d9f0ff','#a0c4ff','#4a9eff']
colors_hi = colors_lo[::-1]

fig, ax = plt.subplots(figsize=(16, 7))

# Shaded bands: (Q05,Q95), (Q10,Q90), (Q25,Q75)
bands = [(0.05,0.95,'#cce5ff',0.25), (0.10,0.90,'#80bfff',0.30), (0.25,0.75,'#3399ff',0.35)]
for lo_q, hi_q, col, alpha in bands:
    ax.fill_between(dates_dt,
                    test_preds[lo_q].flatten(),
                    test_preds[hi_q].flatten(),
                    color=col, alpha=alpha,
                    label=f"{int(lo_q*100)}%–{int(hi_q*100)}% interval")

ax.plot(dates_dt, ys_te,                        color='black',      lw=1.5, label='Actual', zorder=5)
ax.plot(dates_dt, test_preds[0.50].flatten(),   color='darkorange', lw=1.8, ls='--', label='Median (Q50)', zorder=6)

ax.set_title('Quantile Forecast Fan Chart — Test Set', fontsize=14, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Daily Return')
ax.legend(loc='upper left', fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=30); plt.tight_layout(); plt.show()


In [ ]:
# ── Plot 3: Coverage Calibration Plot ────────────────────────────────
expected_coverage = [q * 100 for q in QUANTILES]
observed_coverage = [test_m['coverages'][q] * 100 for q in QUANTILES]

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0,100],[0,100], 'k--', lw=1.5, label='Perfect calibration')
ax.plot(expected_coverage, observed_coverage, 'o-',
        color='royalblue', lw=2, ms=8, label='Model')
for exp, obs, q in zip(expected_coverage, observed_coverage, QUANTILES):
    ax.annotate(f"Q{int(q*100):02d}",
                xy=(exp, obs), xytext=(5, 4), textcoords='offset points', fontsize=9)
ax.set_xlim(0,105); ax.set_ylim(0,105)
ax.set_xlabel('Expected Coverage (%)')
ax.set_ylabel('Observed Coverage (%)')
ax.set_title('Calibration Plot — Quantile Coverage', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── Plot 4: Interval Width & Residuals ───────────────────────────────
interval_width = test_preds[0.95].flatten() - test_preds[0.05].flatten()
residuals      = ys_te - test_preds[0.50].flatten()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Interval width over time
axes[0,0].plot(dates_dt, interval_width, color='steelblue', lw=0.9)
axes[0,0].set_title('90% Prediction Interval Width Over Time')
axes[0,0].set_ylabel('Width'); axes[0,0].set_xlabel('Date')
axes[0,0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[0,0].xaxis.set_major_locator(mdates.MonthLocator(interval=4))
plt.setp(axes[0,0].xaxis.get_majorticklabels(), rotation=30)

# Width distribution
axes[0,1].hist(interval_width, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0,1].axvline(interval_width.mean(), color='red', lw=2, ls='--',
                  label=f'Mean={interval_width.mean():.5f}')
axes[0,1].set_title('Interval Width Distribution'); axes[0,1].legend()

# Residuals
axes[1,0].hist(residuals, bins=60, color='crimson', edgecolor='white', alpha=0.8)
axes[1,0].axvline(0, color='black', lw=2, ls='--', label='Zero')
axes[1,0].axvline(residuals.mean(), color='green', lw=2, ls='-',
                  label=f'Mean={residuals.mean():.5f}')
axes[1,0].set_title('Median Residual Distribution'); axes[1,0].legend()

# Scatter: actual vs median
axes[1,1].scatter(ys_te, test_preds[0.50].flatten(), alpha=0.35, s=12, color='purple')
mn, mx = ys_te.min(), ys_te.max()
axes[1,1].plot([mn,mx],[mn,mx],'r--', lw=2, label='Perfect')
axes[1,1].set_xlabel('Actual'); axes[1,1].set_ylabel('Median Predicted')
axes[1,1].set_title('Actual vs Median Forecast'); axes[1,1].legend()

plt.suptitle('Quantile Forecast Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── Plot 5: Hyperparameter Sensitivity ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, param in zip(axes, ['window_size','units','dropout']):
    grp = results_df.groupby(param)['val_med_RMSE'].mean().reset_index()
    ax.bar(grp[param].astype(str), grp['val_med_RMSE'],
           color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_title(f'Mean Val Median RMSE vs {param}', fontweight='bold')
    ax.set_xlabel(param); ax.set_ylabel('Mean RMSE')
plt.suptitle('Hyperparameter Sensitivity — Quantile Model', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── Plot 6: Next 5-Day Probabilistic Forecast ────────────────────────
X_all_sc = sc.transform(
    df[[c for c in df.columns if c not in [TARGET_COL, DATE_COL]]].values)

seed = X_all_sc[-BEST_WINDOW:]

fc_quantiles = {q: [] for q in QUANTILES}
window_buf = seed.copy()

for step in range(FORECAST_STEPS):
    inp = window_buf[-BEST_WINDOW:][np.newaxis, ...]
    raw = final_model.predict(inp, verbose=0)
    for i, q in enumerate(QUANTILES):
        fc_quantiles[q].append(float(raw[i][0,0]))
    window_buf = np.vstack([window_buf[1:], window_buf[-1]])

last_date = df[DATE_COL].iloc[-1]
fc_dates  = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=FORECAST_STEPS)

# Context
ctx_n   = 60
ctx_dt  = dates_dt[-ctx_n:]
ctx_val = ys_te[-ctx_n:]

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(ctx_dt, ctx_val, color='royalblue', lw=1.8, label='Historical (test)')

ax.fill_between(fc_dates,
                fc_quantiles[0.05], fc_quantiles[0.95],
                alpha=0.20, color='darkorange', label='90% interval (Q05–Q95)')
ax.fill_between(fc_dates,
                fc_quantiles[0.10], fc_quantiles[0.90],
                alpha=0.30, color='darkorange', label='80% interval (Q10–Q90)')
ax.fill_between(fc_dates,
                fc_quantiles[0.25], fc_quantiles[0.75],
                alpha=0.45, color='darkorange', label='50% interval (Q25–Q75)')
ax.plot(fc_dates, fc_quantiles[0.50], 'o--',
        color='darkorange', lw=2.5, ms=8, label='Median forecast')

for fd, fv in zip(fc_dates, fc_quantiles[0.50]):
    ax.annotate(f"{fv:+.4f}",
                xy=(fd, fv), xytext=(0,12), textcoords='offset points',
                ha='center', fontsize=9, color='darkorange', fontweight='bold')

ax.axvline(ctx_dt[-1], color='gray', lw=1.2, ls=':', label='Forecast Start')
ax.set_title('NIFTY50 — 5-Day Probabilistic Return Forecast', fontsize=14, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Predicted Return')
ax.legend(loc='upper left', fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %Y'))
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

print("\n5-Day Probabilistic Forecast:")
print(f"{'Date':>14} {'Q05':>9} {'Q25':>9} {'Median':>9} {'Q75':>9} {'Q95':>9}")
print("-" * 65)
for i, fd in enumerate(fc_dates):
    print(f"{str(fd.date()):>14} "          f"{fc_quantiles[0.05][i]:>+9.5f} "          f"{fc_quantiles[0.25][i]:>+9.5f} "          f"{fc_quantiles[0.50][i]:>+9.5f} "          f"{fc_quantiles[0.75][i]:>+9.5f} "          f"{fc_quantiles[0.95][i]:>+9.5f}")


## 10. Final Summary

In [ ]:
print("\n" + "╔" + "═"*64 + "╗")
print("║" + " NIFTY50 LSTM QUANTILE FORECAST — FINAL SUMMARY ".center(64) + "║")
print("╠" + "═"*64 + "╣")
print(f"║  Best Window Size     : {BEST_WINDOW:<41}║")
print(f"║  Best LSTM Units      : {BEST_UNITS:<41}║")
print(f"║  Best Dropout         : {BEST_DROPOUT:<41}║")
print(f"║  Best Learning Rate   : {BEST_LR:<41}║")
print("╠" + "═"*64 + "╣")
print(f"║  Test Median RMSE     : {test_m['med_RMSE']:<41.6f}║")
print(f"║  Test Median MAE      : {test_m['med_MAE']:<41.6f}║")
print(f"║  Test PICP (90% int.) : {test_m['PICP']:<39.2f} % ║")
print(f"║  Test PINAW           : {test_m['PINAW']:<41.4f}║")
print(f"║  Test Winkler Score   : {test_m['Winkler']:<41.5f}║")
print(f"║  Median Dir. Accuracy : {dir_acc:<39.2f} % ║")
print("╠" + "═"*64 + "╣")
print(f"║  {cov_note[:62]:<62}║")
print(f"║  {rmse_note[:62]:<62}║")
print(f"║  {rel_note[:62]:<62}║")
print("╚" + "═"*64 + "╝")


## 11. Why Quantile (Pinball) Loss?

### The Problem with MSE for Financial Forecasting
Mean Squared Error (MSE) penalises over-predictions and under-predictions **equally** and produces a single point estimate. For financial returns this is insufficient:
- An investor who **over-estimates a loss** will be too conservative but avoid disaster.
- An investor who **under-estimates a loss** may face catastrophic portfolio drawdown.

These errors have **asymmetric real-world costs** — MSE cannot capture this.

### The Quantile (Pinball) Loss Function

For a quantile τ ∈ (0, 1), the pinball loss is:

```
L_τ(y, ŷ) = τ · max(y − ŷ, 0)  +  (1 − τ) · max(ŷ − y, 0)
```

- When **τ > 0.5**: The loss penalises **under-prediction** more heavily → model learns the upper tail.
- When **τ < 0.5**: The loss penalises **over-prediction** more heavily → model learns the lower tail.
- When **τ = 0.5**: Both errors penalised equally → equivalent to Mean Absolute Error → median forecast.

### Why This Is Ideal for Uncertainty Estimation

| Quantile | Meaning | Investment Use |
|---|---|---|
| Q05 / Q10 | Lower tail — worst-case scenario | Stop-loss placement |
| Q50 | Median — most likely outcome | Direction signal |
| Q90 / Q95 | Upper tail — best-case scenario | Profit target |

By training a single LSTM model to simultaneously predict all quantiles (multi-output heads), we get a **full conditional distribution** of future returns — not just a point estimate. This is the key advantage over standard MSE-based LSTM.


## 12. Hyperparameter Tuning Justification

### Selection Criterion: Lowest Validation Median RMSE
The grid search evaluates 54 combinations. Each model trains with **pinball loss** for all 7 quantiles simultaneously. Selection is based on **validation median (Q50) RMSE** — this ensures the central forecast is as accurate as possible while the quantile structure provides calibrated intervals.

### Why Each Parameter Matters for Quantile Models

| Parameter | Role in Quantile Model |
|---|---|
| **Window Size** | Determines historical context. Too short = misses regime patterns; too long = stale data |
| **LSTM Units** | Shared backbone capacity. Must be sufficient to simultaneously learn 7 quantile outputs |
| **Dropout** | Prevents overfitting to training-set volatility regimes. Critical for tail quantiles |
| **Learning Rate** | Controls convergence speed of total pinball loss across all 7 outputs |

### Model Stability Indicators
- **PICP ≈ 90%** confirms the Q05–Q95 interval is well-calibrated (actual coverage close to nominal).
- **Low Winkler Score** means the model produces tight yet reliable intervals.
- Small gap between train and test median RMSE confirms generalisation, not memorisation.


In [ ]:
# ── Hyperparameter Justification — Summary ───────────────────────────────────
print("=" * 72)
print(" QUANTILE MODEL — HYPERPARAMETER JUSTIFICATION ".center(72))
print("=" * 72)
print(f"  Selection Criterion    : LOWEST Validation Median RMSE")
print(f"  Best Window Size       : {BEST_WINDOW}  → captures {BEST_WINDOW}-day market memory")
print(f"  Best LSTM Units        : {BEST_UNITS}   → shared backbone for 7 quantile outputs")
print(f"  Best Dropout           : {BEST_DROPOUT}  → regularises tail quantile overfitting")
print(f"  Best Learning Rate     : {BEST_LR}  → stable pinball loss minimisation")
print()
print(f"  Train Median RMSE  : {tr_m2['med_RMSE']:.6f}")
print(f"  Val   Median RMSE  : {val_m2['med_RMSE']:.6f}")
print(f"  Test  Median RMSE  : {test_m['med_RMSE']:.6f}")
gap = abs(tr_m2['med_RMSE'] - test_m['med_RMSE'])
print(f"  Train–Test Gap     : {gap:.6f}  → {'✅ Minimal overfitting' if gap < 0.003 else '⚠️ Some overfitting'}")
print()
print(f"  Test PICP (90% interval) : {test_m['PICP']:.2f}%  (target ≈ 90%)")
print(f"  Test PINAW               : {test_m['PINAW']:.4f}")
print(f"  Test Winkler Score       : {test_m['Winkler']:.5f}")
print("=" * 72)


## 13. Quantile Validation — Monotonicity Check (CRITICAL)

A valid quantile forecast must satisfy:  **Q10 ≤ Q25 ≤ Q50 ≤ Q75 ≤ Q90**

Violating this is known as **quantile crossing** — it means the lower bound exceeds the upper bound for some samples, which is physically impossible and logically inconsistent.

Causes of crossing:
- Independent output heads (no ordering constraint enforced during training)
- Extreme input sequences that fall outside the training distribution
- Insufficient model capacity for simultaneous multi-quantile learning

If crossing is detected, solutions include:
- **Isotonic regression** post-processing
- **Sorting** the outputs at inference time
- Enforcing monotone constraints via specialised loss functions


In [ ]:
# ── Quantile Validation — Monotonicity Check ─────────────────────────────────
q_order = [0.10, 0.25, 0.50, 0.75, 0.90]
n_samples = len(ys_te)

print("=" * 65)
print(" QUANTILE MONOTONICITY VALIDATION ".center(65))
print("=" * 65)

violations_total = 0
for i in range(len(q_order) - 1):
    q_lo = q_order[i]
    q_hi = q_order[i + 1]
    lo_pred = test_preds[q_lo].flatten()
    hi_pred = test_preds[q_hi].flatten()
    n_violated = np.sum(lo_pred > hi_pred)
    pct = n_violated / n_samples * 100
    violations_total += n_violated
    status = "✅ OK" if n_violated == 0 else f"❌ {n_violated} crossings ({pct:.1f}%)"
    print(f"  Q{int(q_lo*100):02d} ≤ Q{int(q_hi*100):02d} : {status}")

print("-" * 65)
if violations_total == 0:
    print("  ✅ ALL QUANTILE PAIRS VALID — No quantile crossing detected.")
    print("     The model correctly satisfies Q10 ≤ Q25 ≤ Q50 ≤ Q75 ≤ Q90")
    print("     on the entire test set. Intervals are logically consistent.")
else:
    print(f"  ⚠️  Total crossings detected: {violations_total}")
    print("  The model exhibits quantile crossing on some samples.")
    print("  Recommendation: Apply isotonic regression post-processing or")
    print("  sort quantile outputs at inference time.")
    print()
    print("  Quick fix (sorting outputs at inference):")
    print("  sorted_preds = np.sort(np.stack([test_preds[q] for q in QUANTILES], axis=1), axis=1)")
print("=" * 65)

# Visual check: fan for a 30-day window
fig, ax = plt.subplots(figsize=(14, 5))
ctx = 30
dates_sub = pd.to_datetime(ds_te)[-ctx:]
ax.fill_between(dates_sub,
                test_preds[0.10].flatten()[-ctx:],
                test_preds[0.90].flatten()[-ctx:],
                alpha=0.25, color='royalblue', label='Q10–Q90')
ax.fill_between(dates_sub,
                test_preds[0.25].flatten()[-ctx:],
                test_preds[0.75].flatten()[-ctx:],
                alpha=0.35, color='royalblue', label='Q25–Q75')
ax.plot(dates_sub, test_preds[0.50].flatten()[-ctx:],
        color='darkorange', lw=2, label='Q50 Median')
ax.plot(dates_sub, ys_te[-ctx:], color='black', lw=1.5, label='Actual')
ax.set_title('Quantile Ordering Check — Last 30 Test Days', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Return')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 14. PICP & PINAW — Coverage and Sharpness Interpretation

Two key metrics evaluate the quality of probabilistic forecasts:

### PICP — Prediction Interval Coverage Probability
**PICP = % of actual values that fall inside the predicted interval**

For a 90% interval (Q05–Q95), the target coverage is **90%**.

| PICP Value | Interpretation |
|---|---|
| PICP ≥ 90% | ✅ Reliable — interval truly contains 90% of outcomes |
| 80% ≤ PICP < 90% | ⚠️ Slightly under-covering — intervals too tight |
| PICP < 80% | ❌ Poor coverage — model is overconfident |
| PICP >> 95% | ⚠️ Over-covering — intervals too wide (conservative) |

### PINAW — Prediction Interval Normalised Average Width
**PINAW = mean interval width / range of actual values**

| PINAW Value | Interpretation |
|---|---|
| PINAW < 0.3 | ✅ Sharp intervals — high information value |
| 0.3 ≤ PINAW < 0.6 | ⚠️ Moderate width — acceptable uncertainty |
| PINAW ≥ 0.6 | ❌ Wide intervals — low sharpness, limited usefulness |

### The Trade-off
A model can always achieve 100% PICP by making the interval infinitely wide — but that is useless.
The ideal model achieves **high PICP** (reliable) with **low PINAW** (sharp), confirming it has
genuinely learned the data distribution.


In [ ]:
# ── PICP & PINAW Interpretation ──────────────────────────────────────────────
picp  = test_m['PICP']
pinaw = test_m['PINAW']

print("=" * 65)
print(" PICP & PINAW — MODEL RELIABILITY REPORT ".center(65))
print("=" * 65)
print(f"  PICP  (90% interval coverage) : {picp:.2f}%  (target = 90%)")
print(f"  PINAW (normalised width)       : {pinaw:.4f}")
print(f"  Winkler Score                  : {test_m['Winkler']:.5f}")
print()

# PICP verdict
if picp >= 90:
    picp_verdict = f"✅ EXCELLENT — Model reliably captures 90% of outcomes. Interval is trustworthy."
elif picp >= 80:
    picp_verdict = f"⚠️  MARGINAL — {90-picp:.1f}% short of target. Intervals slightly too tight."
else:
    picp_verdict = f"❌ POOR — Coverage {picp:.1f}% well below 90%. Model is overconfident."

# PINAW verdict
if pinaw < 0.3:
    pinaw_verdict = f"✅ SHARP — Intervals are narrow. High information value for decision-making."
elif pinaw < 0.6:
    pinaw_verdict = f"⚠️  MODERATE — Intervals of acceptable width. Some uncertainty remains."
else:
    pinaw_verdict = f"❌ WIDE — Intervals span too broad a range. Limited actionability."

print(f"  Coverage Verdict : {picp_verdict}")
print(f"  Sharpness Verdict: {pinaw_verdict}")
print()

# Combined reliability
if picp >= 85 and pinaw < 0.5:
    rel = "✅ RELIABLE — High coverage + sharp intervals = strong probabilistic model."
elif picp >= 80:
    rel = "⚠️  ACCEPTABLE — Model provides useful uncertainty estimates with minor limitations."
else:
    rel = "❌ NEEDS IMPROVEMENT — Recalibrate model or widen interval quantiles."

print(f"  Overall Reliability: {rel}")
print("=" * 65)

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# PICP bar
targets = [80, 85, 90, 95, 100]
axes[0].barh(['PICP (Model)'], [picp], color='#27ae60' if picp >= 87 else '#e67e22', alpha=0.85, height=0.4)
axes[0].axvline(90, color='red', lw=2, ls='--', label='Target 90%')
axes[0].set_xlim(50, 105)
axes[0].set_title('PICP — Coverage Probability', fontweight='bold')
axes[0].set_xlabel('Coverage (%)')
axes[0].legend()
axes[0].text(picp + 0.5, 0, f'{picp:.1f}%', va='center', fontweight='bold', fontsize=12)

# PINAW bar
axes[1].barh(['PINAW (Model)'], [pinaw], color='#27ae60' if pinaw < 0.4 else '#e74c3c', alpha=0.85, height=0.4)
axes[1].axvline(0.3, color='green', lw=2, ls='--', label='Sharp threshold (0.3)')
axes[1].axvline(0.6, color='red',   lw=2, ls='--', label='Wide threshold (0.6)')
axes[1].set_xlim(0, max(0.8, pinaw + 0.1))
axes[1].set_title('PINAW — Normalised Interval Width', fontweight='bold')
axes[1].set_xlabel('PINAW')
axes[1].legend()
axes[1].text(pinaw + 0.01, 0, f'{pinaw:.4f}', va='center', fontweight='bold', fontsize=12)

plt.suptitle('Model Reliability: Coverage vs Sharpness', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 15. Confidence Interval Interpretation — Q10 to Q90

### What Does the 80% Confidence Band Mean?
The Q10–Q90 band forms an **80% confidence interval** — we expect 80% of actual future returns to fall within this band.

- **Q10 (Lower Bound)**: In a bad scenario, the return is expected to be at least this low. Think of it as a "pessimistic estimate" — only 10% of outcomes would be worse.
- **Q90 (Upper Bound)**: In a good scenario, the return is expected to be at most this high. Only 10% of outcomes would exceed this.
- **Q50 (Median)**: The central forecast — the most likely outcome. 50% of outcomes fall below and 50% above.

### How to Interpret the Fan Chart

| Band | Coverage | Interpretation |
|---|---|---|
| Q25–Q75 (inner, darkest) | 50% | High-confidence core prediction zone |
| Q10–Q90 (middle) | 80% | Standard confidence interval for decision-making |
| Q05–Q95 (outer, lightest) | 90% | Conservative interval for risk management |

### For Investors
- **Narrow band**: Model is confident. Low uncertainty in the forecast.
- **Wide band**: Model is uncertain. Higher risk; consider smaller position sizes.
- **Entire band positive**: Strong buy signal with high confidence.
- **Entire band negative**: Strong sell / avoid signal.
- **Band crosses zero**: Mixed market conditions; use caution.


## 16. Risk Analysis — Investment Decision Framework

**Risk Level** is determined by the width of the Q10–Q90 prediction interval for each forecast day.
- **Narrow interval** (width < mean − 0.5σ) → Market is predictable → **Low Risk**
- **Wide interval** (width > mean + 0.5σ) → High uncertainty → **High Risk**


In [ ]:
# ── Risk Analysis ────────────────────────────────────────────────────────────
# Compute interval widths for 5-day forecast
fc_widths_80 = [fc_quantiles[0.90][i] - fc_quantiles[0.10][i] for i in range(FORECAST_STEPS)]
fc_widths_50 = [fc_quantiles[0.75][i] - fc_quantiles[0.25][i] for i in range(FORECAST_STEPS)]

# Test set width stats for threshold
test_widths_80 = test_preds[0.90].flatten() - test_preds[0.10].flatten()
mean_w  = np.mean(test_widths_80)
std_w   = np.std(test_widths_80)
low_thr = mean_w - 0.5 * std_w
hi_thr  = mean_w + 0.5 * std_w

BULL_THRESHOLD =  0.003
BEAR_THRESHOLD = -0.003

print("=" * 90)
print(" 5-DAY RISK ANALYSIS & INVESTMENT RECOMMENDATION ".center(90))
print("=" * 90)
print(f"  {'Day':<4} {'Date':<14} {'Median':>10} {'Q10':>10} {'Q90':>10} {'Width':>10} "
      f"{'Risk':>12} {'Signal':>11}")
print("-" * 90)

for i, (bd, fw) in enumerate(zip(fc_dates, fc_widths_80)):
    med_v = fc_quantiles[0.50][i]
    q10_v = fc_quantiles[0.10][i]
    q90_v = fc_quantiles[0.90][i]

    if fw <= low_thr:
        risk_label = "✅ LOW RISK"
        invest_ok  = True
    elif fw >= hi_thr:
        risk_label = "⚠️ HIGH RISK"
        invest_ok  = False
    else:
        risk_label = "🟡 MOD RISK"
        invest_ok  = None

    if med_v > BULL_THRESHOLD:
        signal = "🟢 BULLISH"
    elif med_v < BEAR_THRESHOLD:
        signal = "🔴 BEARISH"
    else:
        signal = "🟡 NEUTRAL"

    print(f"  {i+1:<4} {str(bd.date()):<14} {med_v:>+10.5f} {q10_v:>+10.5f} {q90_v:>+10.5f} "
          f"{fw:>10.5f} {risk_label:>12} {signal:>11}")

print("=" * 90)
print(f"\n  Reference: mean test interval width = {mean_w:.5f} ± {std_w:.5f}")
print(f"  Low Risk threshold  : width ≤ {low_thr:.5f}")
print(f"  High Risk threshold : width ≥ {hi_thr:.5f}")
print()

# Final investment recommendation
n_low  = sum(1 for w in fc_widths_80 if w <= low_thr)
n_high = sum(1 for w in fc_widths_80 if w >= hi_thr)
median_sum = sum(fc_quantiles[0.50])

if n_high >= 3:
    print("  📛 FINAL RECOMMENDATION: ⚠️  RISKY MARKET CONDITIONS")
    print("     Most forecast days show HIGH uncertainty. Reduce position sizes.")
    print("     Wait for clearer market direction before committing capital.")
elif n_low >= 3 and median_sum > 0:
    print("  📗 FINAL RECOMMENDATION: ✅ RELATIVELY SAFE TO INVEST")
    print("     Most days show LOW uncertainty + POSITIVE median returns.")
    print("     Favourable conditions for long positions with defined stop-loss.")
elif n_low >= 3 and median_sum < 0:
    print("  📕 FINAL RECOMMENDATION: ✅ LOW RISK BUT BEARISH")
    print("     Low uncertainty, but median returns are negative.")
    print("     Consider short exposure or sitting out. Not ideal for longs.")
else:
    print("  📒 FINAL RECOMMENDATION: 🟡 MIXED CONDITIONS")
    print("     Mixed risk profile. Exercise caution, use smaller position sizes.")
    print("     Monitor closely — market is in a transition phase.")
print()
print("  ⚠️  This is a model output for educational purposes only.")
print("     Always consult a SEBI-registered advisor before investing.")

# Plot interval widths for forecast days
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

day_labels = [str(bd.date()) for bd in fc_dates]
bar_colors = ['#27ae60' if w <= low_thr else '#e74c3c' if w >= hi_thr else '#f39c12'
              for w in fc_widths_80]

axes[0].bar(day_labels, fc_widths_80, color=bar_colors, edgecolor='white', alpha=0.87)
axes[0].axhline(mean_w,  color='black',  lw=1.5, ls='--', label=f'Mean = {mean_w:.4f}')
axes[0].axhline(low_thr, color='green',  lw=1.5, ls=':',  label=f'Low Risk ≤ {low_thr:.4f}')
axes[0].axhline(hi_thr,  color='red',    lw=1.5, ls=':',  label=f'High Risk ≥ {hi_thr:.4f}')
axes[0].set_title('Forecast Interval Width (Risk Level)', fontweight='bold')
axes[0].set_ylabel('Q10–Q90 Width')
axes[0].set_xlabel('Forecast Date')
axes[0].legend(fontsize=8)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30)

# Forecast fan
axes[1].fill_between(range(FORECAST_STEPS),
                     fc_quantiles[0.10], fc_quantiles[0.90],
                     alpha=0.25, color='royalblue', label='80% CI (Q10–Q90)')
axes[1].fill_between(range(FORECAST_STEPS),
                     fc_quantiles[0.25], fc_quantiles[0.75],
                     alpha=0.40, color='royalblue', label='50% CI (Q25–Q75)')
axes[1].plot(range(FORECAST_STEPS), fc_quantiles[0.50], 'o-',
             color='darkorange', lw=2, ms=8, label='Median Forecast')
axes[1].axhline(0, color='black', lw=0.8, ls='--')
axes[1].set_xticks(range(FORECAST_STEPS))
axes[1].set_xticklabels(day_labels, rotation=30, fontsize=9)
axes[1].set_title('5-Day Probabilistic Forecast with Risk Zones', fontweight='bold')
axes[1].set_ylabel('Predicted Return')
axes[1].legend(fontsize=9)

plt.suptitle('Risk Analysis — 5-Day Forecast', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 17. Sector Analysis — Identifying the Driving Force

Understanding which sector will drive NIFTY50 over the next 5 days enhances
the interpretability of our probabilistic forecast. If the top correlated sector
shows a positive trend, we expect the model's upper quantiles to dominate.


In [ ]:
# ── Sector Analysis (Quantile Notebook) ─────────────────────────────────────
sector_keywords = ['bank', 'it', 'pharma', 'auto', 'fmcg', 'metal', 'energy',
                   'infra', 'realty', 'media', 'finance']

sector_cols = [c for c in df.columns
               if any(kw in c.lower() for kw in sector_keywords)
               and c not in [TARGET_COL, DATE_COL]]

if len(sector_cols) == 0:
    feat_cols_all = [c for c in df.columns if c not in [TARGET_COL, DATE_COL]]
    corr_series   = df[feat_cols_all].corrwith(df[TARGET_COL]).abs().sort_values(ascending=False)
    sector_cols   = corr_series.head(8).index.tolist()

corr_vals  = df[sector_cols].corrwith(df[TARGET_COL]).sort_values(ascending=False)
top_sector = corr_vals.index[0]
top_corr   = corr_vals.iloc[0]

# Recent sector trend
recent_target   = df[TARGET_COL].values[-20:]
recent_top_sect = df[top_sector].values[-20:]
recent_sector_avg = np.mean(recent_top_sect[-5:])

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(20), recent_target,   label='NIFTY50 Return', color='royalblue', lw=1.8)
ax.plot(range(20), recent_top_sect, label=f'{top_sector[:30]} (Top Driver)',
        color='darkorange', lw=1.5, ls='--')
ax.axhline(0, color='black', lw=0.6, ls=':')
ax.set_title(f'Recent Market: NIFTY50 vs {top_sector[:30]}', fontweight='bold')
ax.set_xlabel('Days (most recent 20)')
ax.set_ylabel('Return / Feature Value (normalised)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\n  Top correlated driver : '{top_sector}' (corr = {top_corr:+.4f})")
print(f"  5-day average of top driver (most recent) : {recent_sector_avg:.5f}")
print()

# Link to quantile forecast
q50_avg = np.mean(fc_quantiles[0.50])
q10_avg = np.mean(fc_quantiles[0.10])
q90_avg = np.mean(fc_quantiles[0.90])

print("  Sector-Quantile Link:")
if top_corr > 0:
    if recent_sector_avg > 0:
        print(f"  ✅ Top driver shows POSITIVE recent trend → supports upward median forecast (Q50 avg = {q50_avg:+.5f})")
        print(f"     Upper bound (Q90 avg = {q90_avg:+.5f}) → optimistic scenario intact.")
    else:
        print(f"  ⚠️  Top driver shows NEGATIVE recent trend → downward pressure on NIFTY50")
        print(f"     Lower bound (Q10 avg = {q10_avg:+.5f}) is the risk scenario to watch.")
else:
    print(f"  ℹ️  Top driver shows INVERSE correlation — monitor for divergence signals.")


## 18. Final Structured Output Table

Combining point forecast (median), uncertainty bounds, market signal, and risk level into a single decision-ready table.


In [ ]:
# ── Final Structured Output Table ───────────────────────────────────────────
BULL_THRESHOLD =  0.003
BEAR_THRESHOLD = -0.003

# Re-compute widths
fc_widths_80 = [fc_quantiles[0.90][i] - fc_quantiles[0.10][i] for i in range(FORECAST_STEPS)]
test_widths_80 = test_preds[0.90].flatten() - test_preds[0.10].flatten()
mean_w  = np.mean(test_widths_80)
std_w   = np.std(test_widths_80)
low_thr = mean_w - 0.5 * std_w
hi_thr  = mean_w + 0.5 * std_w

print("=" * 108)
print(" NIFTY50 — 5-DAY PROBABILISTIC FORECAST — COMPLETE OUTPUT ".center(108))
print("=" * 108)
print(f"  {'Day':<4} {'Date':<14} {'Median%':>9} {'Lower(Q10)':>11} {'Upper(Q90)':>11} "
      f"{'Width':>9} {'Signal':>11} {'Risk':>12}")
print("-" * 108)

for i, (bd, fw) in enumerate(zip(fc_dates, fc_widths_80)):
    med_v  = fc_quantiles[0.50][i]
    q10_v  = fc_quantiles[0.10][i]
    q90_v  = fc_quantiles[0.90][i]
    pct    = med_v * 100

    if med_v > BULL_THRESHOLD:
        signal = "🟢 BULLISH"
    elif med_v < BEAR_THRESHOLD:
        signal = "🔴 BEARISH"
    else:
        signal = "🟡 NEUTRAL"

    if fw <= low_thr:
        risk = "✅ LOW RISK"
    elif fw >= hi_thr:
        risk = "⚠️ HIGH RISK"
    else:
        risk = "🟡 MOD RISK"

    print(f"  {i+1:<4} {str(bd.date()):<14} {pct:>+8.3f}%  {q10_v:>+11.5f}  {q90_v:>+11.5f}  "
          f"{fw:>9.5f}  {signal:<12} {risk}")

print("=" * 108)
print()
print("  Column Definitions:")
print("  Median%   : Central forecast (Q50) expressed as % return")
print("  Lower     : Q10 — worst-case (pessimistic) expected return")
print("  Upper     : Q90 — best-case (optimistic) expected return")
print("  Width     : Uncertainty span (Q90 − Q10) — higher = more uncertain")
print("  Signal    : Bullish >+0.3% | Bearish <-0.3% | Neutral otherwise")
print("  Risk      : Based on interval width vs historical average")


## 19. Final Conclusion — Probabilistic NIFTY50 Forecasting

### What This Notebook Achieves

This project implements a complete **uncertainty-aware LSTM forecasting pipeline** for NIFTY50 daily returns:

1. **Quantile Regression LSTM** — simultaneous prediction of Q05, Q10, Q25, Q50, Q75, Q90, Q95 using pinball loss.
2. **Systematic hyperparameter tuning** — 54 combinations evaluated on validation median RMSE.
3. **Probabilistic evaluation** — PICP (coverage), PINAW (sharpness), Winkler Score.
4. **Quantile validation** — monotonicity check ensuring Q10 ≤ Q50 ≤ Q90 across all samples.
5. **Risk analysis** — each forecast day labelled by uncertainty level (Low/Moderate/High Risk).
6. **Sector linkage** — most influential market driver identified and linked to the 5-day forecast.
7. **Structured decision table** — actionable output combining signal, bounds, and risk.

### Why Quantile LSTM is Superior to Point LSTM
- Point LSTM gives **one number** (e.g., +0.5%) — investor cannot assess confidence.
- Quantile LSTM gives **a range** (e.g., Q10=−0.3%, Q50=+0.5%, Q90=+1.2%) — investor can size positions based on the full uncertainty profile.
- This is the difference between "the market will go up" and "the market will likely go up, but there is a 10% chance it falls to −0.3% and a 10% chance it rallies to +1.2%."

### Combined Point + Quantile System
Used together:
- **Point model** → primary direction signal
- **Quantile model** → confidence interval + risk level
- Agreement between both → high-confidence trade; divergence → reduce exposure

### Academic & Industry Relevance
This framework aligns with state-of-the-art probabilistic forecasting used in:
- Quantitative hedge funds (interval-based position sizing)
- Risk management desks (Value-at-Risk from quantile tails)
- Robo-advisors (uncertainty-aware portfolio construction)


## 20. Interview-Ready Statements

> **"I implemented a quantile regression LSTM for NIFTY50 return forecasting. Instead of minimising MSE to get a single point estimate, I used pinball loss to simultaneously train 7 output heads — one for each quantile (Q05 to Q95). This gives a full conditional distribution of future returns, which is far more useful for financial decision-making than a single predicted value."**

> **"The key metric I used for model selection is validation median RMSE — this ensures the central forecast is accurate. I then validated the interval quality using PICP (which checks whether 90% of actuals truly fall within the Q05–Q95 band) and PINAW (which measures how tight or wide the intervals are). A good model achieves high PICP with low PINAW simultaneously."**

> **"One important validation step I always include is quantile monotonicity checking — verifying that Q10 ≤ Q25 ≤ Q50 ≤ Q75 ≤ Q90 for all test samples. Quantile crossing is a sign of model inconsistency and would make the output logically invalid for risk management purposes."**
